# Random Forest — The Library Version

Same model via scikit-learn, verifying the scratch build — including its built-in OOB score and stabilized feature importances (README §5).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("data/mushroom_data.csv")
X = df[["cap_cm", "spore_density"]].values
y = (df.label == "poisonous").astype(int).values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=44)

tree = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
forest = RandomForestClassifier(n_estimators=200, max_features=1,      # our two randomness sources
                                oob_score=True, random_state=0).fit(Xtr, ytr)

print(f"single tree : {accuracy_score(yte, tree.predict(Xte)):.0%}")
print(f"forest (200): {accuracy_score(yte, forest.predict(Xte)):.0%}")
print(f"OOB score   : {forest.oob_score_:.0%}   <- the free exam (README §3.4), one flag away")
print(f"feature importances: cap_cm={forest.feature_importances_[0]:.2f}, "
      f"spore_density={forest.feature_importances_[1]:.2f}  (averaged over 200 trees — steadier than lesson 05's)")

single tree : 82%
forest (200): 90%
OOB score   : 91%   <- the free exam (README §3.4), one flag away
feature importances: cap_cm=0.48, spore_density=0.52  (averaged over 200 trees — steadier than lesson 05's)


In [2]:
# The lesson 05 instability demo, resolved: the FOREST's first tree changes across subsamples,
# but the FOREST's predictions barely move.
rng = np.random.default_rng(1)
preds = []
for i in range(5):
    keep = rng.choice(len(Xtr), int(0.9*len(Xtr)), replace=False)
    f = RandomForestClassifier(n_estimators=100, max_features=1, random_state=i).fit(Xtr[keep], ytr[keep])
    preds.append(f.predict(Xte))
agree = np.mean([np.mean(preds[i] == preds[j]) for i in range(5) for j in range(i+1, 5)])
print(f"Across five 90% subsamples, forests agree on {agree:.0%} of test predictions.")
print("Lesson 05's root-flipping chaos is still happening INSIDE — the vote just doesn't care.")

Across five 90% subsamples, forests agree on 98% of test predictions.
Lesson 05's root-flipping chaos is still happening INSIDE — the vote just doesn't care.


**The takeaway:** `RandomForestClassifier` = our Blocks 3–6 with parallelism and decades of tuning. `max_features` and `bootstrap` are the two randomness dials you hand-built; `oob_score=True` is Block 9 for free. No magic — engineered disagreement, voted.